# LAB 4: Neural Sparse Search com SPLADE Multilíngue (OpenSearch 3.x)
## Modelo pré-treinado `opensearch-neural-sparse-encoding-multilingual-v1` via ML Commons

**Objetivo:** Implementar **Neural Sparse Search** no OpenSearch 3.x usando o modelo **pré-treinado oficial multilíngue** da Amazon/OpenSearch e ingest/search **pipelines neural sparse**. Compararemos com BM25 e kNN dense (BGE-M3, do LAB1) sobre o corpus jurídico TCU 2026.

**Modelo:** `amazon/neural-sparse/opensearch-neural-sparse-encoding-multilingual-v1`  
**Dataset:** `corpus_juridico_aula4_v2.json` (1.100 acórdãos TCU 2026)

**Referências:**
- OpenSearch Docs: *Neural sparse search using pipelines* — <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-sparse-with-pipelines/>
- OpenSearch Docs: *Neural sparse search* — <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-sparse-search/>
- FORMAL et al. **SPLADE**. arXiv:2107.05720, 2021.

**Pré-requisitos:** LAB1 executado (ML Commons habilitado, índice `corpus_juridico_aula4` populado).

## 1. Setup

In [ ]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'pandas', 'matplotlib', 'seaborn',
            'langfuse>=2.36.0', 'python-dotenv', 'tqdm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import json, os, time, warnings
from opensearchpy import OpenSearch
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from dotenv import load_dotenv
from tqdm import tqdm
warnings.filterwarnings('ignore')
load_dotenv()

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, ssl_show_warn=False, http_compress=True,
)
print(f"OpenSearch {client.info()['version']['number']} OK")

## 2. Habilitar Neural Sparse no Cluster

In [ ]:
# Permitir uso de modelos pré-treinados locais (não-remote) no ML Commons
settings_body = {
    'persistent': {
        'plugins.ml_commons.only_run_on_ml_node': 'false',
        'plugins.ml_commons.allow_registering_model_via_url': 'true',
        'plugins.ml_commons.allow_registering_model_via_local_file': 'true',
        'plugins.ml_commons.native_memory_threshold': '99',
        'plugins.ml_commons.jvm_heap_memory_threshold': '85'
    }
}
resp = client.transport.perform_request('PUT', '/_cluster/settings', body=settings_body)
print('Cluster preparado para modelos pré-treinados Neural Sparse.')

## 3. Registrar Modelo Pré-Treinado Multilíngue

O OpenSearch ML Commons mantém um catálogo de modelos pré-treinados. O nome registrado segue o padrão:
`amazon/neural-sparse/opensearch-neural-sparse-encoding-multilingual-v1` (versão 1.0.0).

In [ ]:
MODEL_NAME = 'amazon/neural-sparse/opensearch-neural-sparse-encoding-multilingual-v1'
MODEL_VERSION = '1.0.0'

model_group_body = {
    'name': 'aula4_neural_sparse_group',
    'description': 'Modelos Neural Sparse para LAB4 da Aula 4',
    'access_mode': 'public'
}
try:
    g = client.transport.perform_request(
        'POST', '/_plugins/_ml/model_groups/_register', body=model_group_body)
    model_group_id = g['model_group_id']
except Exception as e:
    # Recuperar grupo existente
    search = client.transport.perform_request(
        'POST', '/_plugins/_ml/model_groups/_search',
        body={'query': {'match': {'name': 'aula4_neural_sparse_group'}}, 'size': 1})
    model_group_id = search['hits']['hits'][0]['_id']
print(f'model_group_id: {model_group_id}')

register_body = {
    'name':          MODEL_NAME,
    'version':       MODEL_VERSION,
    'model_group_id': model_group_id,
    'model_format':  'TORCH_SCRIPT'
}
r = client.transport.perform_request(
    'POST', '/_plugins/_ml/models/_register?deploy=true', body=register_body)
task_id = r['task_id']
print(f'Tarefa de registro: {task_id}')

sparse_model_id = None
for _ in range(120):
    s = client.transport.perform_request('GET', f'/_plugins/_ml/tasks/{task_id}')
    if s.get('state') == 'COMPLETED':
        sparse_model_id = s.get('model_id')
        break
    if s.get('state') == 'FAILED':
        raise RuntimeError(f'Registro falhou: {s}')
    time.sleep(3)

print(f'Neural Sparse model_id: {sparse_model_id}')

## 4. Smoke-test do Modelo Sparse

A saída de um Neural Sparse encoder é um mapa **token → peso** (vetor esparso interpretável).

In [ ]:
test_body = {'text_docs': ['operação de crédito com garantia da União']}
resp = client.transport.perform_request(
    'POST', f'/_plugins/_ml/_predict/sparse_encoding/{sparse_model_id}', body=test_body)
tokens = resp['inference_results'][0]['output'][0]['dataAsMap']['response'][0]
print(f'Tokens não-zero: {len(tokens)}')
top = sorted(tokens.items(), key=lambda kv: kv[1], reverse=True)[:15]
for tk, w in top:
    print(f'  {tk:25} {w:.4f}')

## 5. Ingest Pipeline `sparse_encoding`

Gera o vetor esparso para o campo `conteudo` e o grava em `sparse_embedding`.

In [ ]:
INGEST_PIPELINE_SPARSE = 'pipeline_neural_sparse_ingest'
ingest_body = {
    'description': 'Gera vetor esparso (SPLADE multilíngue) durante ingestão',
    'processors': [
        {
            'sparse_encoding': {
                'model_id': sparse_model_id,
                'field_map': {
                    'conteudo': 'sparse_embedding'
                }
            }
        }
    ]
}
client.transport.perform_request('PUT', f'/_ingest/pipeline/{INGEST_PIPELINE_SPARSE}', body=ingest_body)
print(f'Ingest pipeline criado: {INGEST_PIPELINE_SPARSE}')

## 6. Criar Índice Neural Sparse (`rank_features`)

O campo que armazena o vetor esparso é do tipo `rank_features` (mapa token→peso).

In [ ]:
INDEX_SPARSE = 'corpus_juridico_aula4_sparse'

index_body = {
    'settings': {
        'index': {
            'default_pipeline':   INGEST_PIPELINE_SPARSE,
            'number_of_shards':   1,
            'number_of_replicas': 0
        },
        'analysis': {
            'analyzer': {'portuguese_analyzer': {'type': 'standard', 'stopwords': '_portuguese_'}}
        }
    },
    'mappings': {
        'properties': {
            'id':       {'type': 'keyword'},
            'tipo':     {'type': 'keyword'},
            'titulo':   {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'conteudo': {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'metadata': {'type': 'object'},
            'sparse_embedding': {'type': 'rank_features'}
        }
    }
}
if client.indices.exists(index=INDEX_SPARSE):
    client.indices.delete(index=INDEX_SPARSE)
client.indices.create(index=INDEX_SPARSE, body=index_body)
print(f'Índice criado: {INDEX_SPARSE}')

## 7. Indexar o Corpus Jurídico Enriquecido (1.100 acórdãos TCU 2026)

In [ ]:
CORPUS_PATH = '../datasets/corpus_juridico_aula4_v2.json'
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)
print(f'Corpus: {len(corpus)} documentos')

BATCH = 15  # encoding sparse é mais pesado; lotes menores ajudam
for start in tqdm(range(0, len(corpus), BATCH), desc='Ingest sparse'):
    sub = corpus[start:start + BATCH]
    bulk = []
    for d in sub:
        bulk.append({'index': {'_index': INDEX_SPARSE, '_id': d['id']}})
        bulk.append({
            'id': d['id'], 'tipo': d.get('tipo', ''),
            'titulo': d.get('titulo', ''),
            'conteudo': d.get('conteudo', ''),
            'metadata': d.get('metadata', {})
        })
    client.bulk(body=bulk, request_timeout=180)
client.indices.refresh(index=INDEX_SPARSE)
print(f'Total indexado: {client.count(index=INDEX_SPARSE)["count"]}')

## 8. Search com `neural_sparse` Query

A query `neural_sparse` aceita o texto bruto e o `model_id`. O OpenSearch faz o encoding em tempo de consulta.

In [ ]:
def search_neural_sparse(query: str, k: int = 5):
    body = {
        'size': k,
        'query': {
            'neural_sparse': {
                'sparse_embedding': {
                    'query_text': query,
                    'model_id':   sparse_model_id
                }
            }
        },
        '_source': ['id', 'titulo']
    }
    t0 = time.perf_counter()
    r = client.search(index=INDEX_SPARSE, body=body)
    lat_ms = (time.perf_counter() - t0) * 1000
    return r['hits']['hits'], lat_ms

hits, lat = search_neural_sparse('operação de crédito com garantia da União', k=5)
print(f'Latência: {lat:.1f} ms')
for h in hits:
    print(f"  [{h['_score']:.4f}] {h['_source']['titulo'][:90]}")

## 9. Comparação Neural Sparse vs BM25 vs Dense kNN (BGE-M3)

Reutilizamos `lab1_config.json` e o índice dense do LAB1 para o BGE-M3.

In [ ]:
with open('lab1_config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
INDEX_DENSE   = cfg['index_name']
DENSE_MODEL_ID = cfg['model_id']

def search_bm25(query: str, k: int = 5):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_DENSE,
                      body={'size': k, 'query': {'match': {'conteudo': {'query': query}}},
                            '_source': ['id', 'titulo']})
    return r['hits']['hits'], (time.perf_counter() - t0) * 1000

def search_neural_dense(query: str, k: int = 5):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_DENSE,
                      body={'size': k,
                            'query': {'neural': {'knn_vector': {
                                'query_text': query, 'model_id': DENSE_MODEL_ID, 'k': k}}},
                            '_source': ['id', 'titulo']})
    return r['hits']['hits'], (time.perf_counter() - t0) * 1000

with open('../datasets/queries_avaliacao_aula4.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)
print(f'Queries: {len(queries)}')

In [ ]:
def mrr(r, rel, k=10):
    for i, d in enumerate(r[:k], 1):
        if d in rel: return 1.0 / i
    return 0.0

def recall(r, rel, k=10):
    if not rel: return 0.0
    return len(set(r[:k]) & set(rel)) / len(rel)

def ndcg(r, rel, k=10):
    dcg  = sum((1.0 if d in rel else 0.0) / np.log2(i+1) for i, d in enumerate(r[:k], 1))
    idcg = sum(1.0 / np.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg/idcg if idcg > 0 else 0.0

K = 10
rows = []
for q in tqdm(queries):
    rel = q['documentos_relevantes']
    for nome, fn in [('BM25',        search_bm25),
                      ('Dense_BGE_M3', search_neural_dense),
                      ('Neural_Sparse', search_neural_sparse)]:
        hits, lat = fn(q['texto'], k=K)
        ids = [h['_source']['id'] for h in hits]
        rows.append({'query_id': q['id'], 'estrategia': nome,
                     'MRR@10':    mrr(ids, rel, K),
                     'Recall@10': recall(ids, rel, K),
                     'NDCG@10':   ndcg(ids, rel, K),
                     'latencia_ms': lat})

df = pd.DataFrame(rows)
df.head()

In [ ]:
agg = df.groupby('estrategia')[['MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']].mean().round(4)
print('Médias por estratégia:')
print(agg)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
agg[['MRR@10', 'Recall@10', 'NDCG@10']].plot.bar(ax=ax[0])
ax[0].set_title('Qualidade média'); ax[0].set_ylim(0, 1)
ax[0].grid(axis='y', alpha=0.3); ax[0].tick_params(axis='x', rotation=20)
agg[['latencia_ms']].plot.bar(ax=ax[1], color='#DD8452')
ax[1].set_title('Latência média (ms)'); ax[1].grid(axis='y', alpha=0.3)
ax[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig('lab4_comparativo.png', dpi=140); plt.show()

## 10. Registrar no LangFuse

In [ ]:
from langfuse import Langfuse
LANGFUSE_HOST       = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')
try:
    lf = Langfuse(public_key=LANGFUSE_PUBLIC_KEY, secret_key=LANGFUSE_SECRET_KEY, host=LANGFUSE_HOST)
    trace = lf.trace(
        name='Aula4-LAB4-Neural-Sparse-vs-Dense-vs-BM25',
        metadata={'aula': '4', 'lab': '4', 'corpus': 'TCU 2026 (1.100 acórdãos)',
                  'sparse_model': MODEL_NAME, 'dense_model': cfg.get('embed_model'),
                  'n_queries': len(queries)}
    )
    for nome, m in agg.iterrows():
        for col in ['MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']:
            trace.score(name=f'{col}_{nome}', value=float(m[col]),
                        comment=f'Média {col} para {nome}')
    lf.flush()
    print(f'Trace: {LANGFUSE_HOST}/traces/{trace.id}')
except Exception as e:
    print(f'LangFuse indisponível: {e}')

## 11. Exportar Configuração + Resultados (para LAB6)

In [ ]:
cfg_out = {
    'sparse_index':     INDEX_SPARSE,
    'sparse_model_id':  sparse_model_id,
    'sparse_model_name': MODEL_NAME,
    'ingest_pipeline_sparse': INGEST_PIPELINE_SPARSE
}
with open('lab4_config.json', 'w', encoding='utf-8') as f:
    json.dump(cfg_out, f, ensure_ascii=False, indent=2)
df.to_csv('lab4_resultados.csv', index=False)
print('lab4_config.json e lab4_resultados.csv gravados.')

## 12. Análise — Interpretabilidade do SPLADE

Cada documento indexado tem agora um vetor `sparse_embedding` token→peso. Inspecionar um documento mostra **quais tokens o modelo considerou relevantes**, com pesos quantitativos:

In [ ]:
doc = client.get(index=INDEX_SPARSE, id=corpus[0]['id'])
tokens_doc = doc['_source'].get('sparse_embedding', {})
print(f"Título: {doc['_source']['titulo'][:90]}")
print(f"Tokens não-zero: {len(tokens_doc)}")
print('Top-20 tokens com maior peso (interpretabilidade SPLADE):')
top = sorted(tokens_doc.items(), key=lambda kv: kv[1], reverse=True)[:20]
for tk, w in top:
    print(f'  {tk:25} {w:.4f}')

## Conclusões

1. **Neural Sparse Search** combina o **inverted index** (rápido, BM25-like) com a **expansão semântica** dos transformers — captura sinônimos sem custo de embedding denso.
2. O modelo `opensearch-neural-sparse-encoding-multilingual-v1` é **multilíngue** e funciona bem em português.
3. **Trade-off**: Sparse tende a ser **mais rápido em retrieval** que dense kNN (menos espaço a varrer), mas **mais lento na ingestão** (encoding pesado).

## Referências (ABNT)

OPENSEARCH PROJECT. **Neural sparse search using pipelines**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-sparse-with-pipelines/>.

OPENSEARCH PROJECT. **Neural sparse search**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-sparse-search/>.

FORMAL, T. et al. **SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking**. SIGIR, 2021. arXiv:2107.05720.

DEVLIN, J. et al. **BERT: Pre-training of Deep Bidirectional Transformers**. NAACL, 2019.

AMAZON OPENSEARCH SERVICE. **opensearch-neural-sparse-encoding-multilingual-v1 (model card)**, 2024.